# Proyecto final de Analítica de Negocios: entregas Olist
**Pregunta:** ¿pueden las características conocidas al comprar identificar pedidos que terminarán después de la fecha prometida?  
**Modelos:** árbol de decisión (clasificación) y K-Means (segmentación).  
Fuente original: https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce . Copia educativa comprimida: https://github.com/infobarbosa/dataset-olist . Licencia de la copia: CC BY-NC-SA 4.0. Ejecutar las celdas en orden. No se incorporan cifras de modelos no ejecutados.

In [ ]:
!pip -q install pandas numpy scikit-learn matplotlib seaborn requests

In [ ]:
from pathlib import Path
import pandas as pd, numpy as np, matplotlib.pyplot as plt, seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.cluster import KMeans
from sklearn.metrics import classification_report, confusion_matrix, average_precision_score, roc_auc_score, precision_recall_curve, silhouette_score
import requests
BASE='https://raw.githubusercontent.com/infobarbosa/dataset-olist/main/data/'
names=['olist_orders_dataset','olist_order_items_dataset','olist_customers_dataset','olist_sellers_dataset','olist_products_dataset','olist_order_payments_dataset']
Path('data').mkdir(exist_ok=True)
for name in names:
    dest=Path('data')/(name+'.csv.gz')
    if not dest.exists():
        r=requests.get(BASE+dest.name,timeout=120); r.raise_for_status(); dest.write_bytes(r.content)
tables={n:pd.read_csv(Path('data')/(n+'.csv.gz')) for n in names}
for n,t in tables.items(): print(n,t.shape,'duplicados ID',t.duplicated().sum())

## Definición y preparación
Una fila representa una orden. Se agregan ítems y pagos antes del cruce para evitar multiplicación de filas. La etiqueta solo existe en órdenes entregadas con fechas reales y prometidas. Los predictores se limitan a datos conocidos al hacer la compra; por ello no se usan aprobación, despacho, entrega real, reseñas ni estado final.

In [ ]:
orders=tables['olist_orders_dataset'].copy()
for c in ['order_purchase_timestamp','order_delivered_customer_date','order_estimated_delivery_date']:
    orders[c]=pd.to_datetime(orders[c],errors='coerce')
items=tables['olist_order_items_dataset']
products=tables['olist_products_dataset']
sellers=tables['olist_sellers_dataset']
customers=tables['olist_customers_dataset']
item=items.merge(products[['product_id','product_weight_g','product_length_cm','product_height_cm','product_width_cm']],on='product_id',how='left',validate='many_to_one')
item=item.merge(sellers[['seller_id','seller_state']],on='seller_id',how='left',validate='many_to_one')
agg=item.groupby('order_id').agg(n_items=('order_item_id','count'),n_sellers=('seller_id','nunique'),price_sum=('price','sum'),freight_sum=('freight_value','sum'),weight_mean=('product_weight_g','mean'),weight_max=('product_weight_g','max'),length_mean=('product_length_cm','mean'),height_mean=('product_height_cm','mean'),width_mean=('product_width_cm','mean'),seller_state_first=('seller_state','first')).reset_index()
df=orders.merge(customers[['customer_id','customer_state']],on='customer_id',how='left',validate='many_to_one').merge(agg,on='order_id',how='left',validate='one_to_one')
assert df.order_id.is_unique and len(df)==len(orders)
df['cross_state']=(df.customer_state!=df.seller_state_first).astype('float')
df.loc[df.customer_state.isna()|df.seller_state_first.isna(),'cross_state']=np.nan
df['purchase_month']=df.order_purchase_timestamp.dt.month
df['purchase_weekday']=df.order_purchase_timestamp.dt.dayofweek
df['promised_days']=(df.order_estimated_delivery_date-df.order_purchase_timestamp).dt.total_seconds()/86400
eligible=df[(df.order_status=='delivered') & df.order_purchase_timestamp.notna() & df.order_delivered_customer_date.notna() & df.order_estimated_delivery_date.notna() & (df.promised_days>=0)].copy()
eligible['late']=(eligible.order_delivered_customer_date>eligible.order_estimated_delivery_date).astype(int)
print('Órdenes originales:',len(df),'Elegibles:',len(eligible),'Excluidas:',len(df)-len(eligible))
print('Tasa tardía:',eligible.late.mean())
print('Faltantes principales:'); print(eligible[['customer_state','price_sum','freight_sum','weight_mean','seller_state_first']].isna().mean())

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4))
eligible.late.value_counts(normalize=True).sort_index().plot.bar(ax=ax[0],title='Entregas a tiempo y tardías')
eligible.groupby('purchase_month').late.mean().plot(ax=ax[1],marker='o',title='Tasa tardía por mes de compra')
plt.tight_layout(); plt.savefig('eda.png',dpi=160); plt.show()

## Árbol y validación temporal
El 80 % más antiguo se usa para entrenamiento y el 20 % más reciente para prueba. Los faltantes y categorías desconocidas se tratan dentro del pipeline. La fecha prometida puede revisarse como predictor porque se comunica al comprar, pero este enfoque estima incumplimiento de promesa y no duración absoluta. Se compara con la regla de predecir siempre «a tiempo».

In [ ]:
features=['customer_state','seller_state_first','cross_state','n_items','n_sellers','price_sum','freight_sum','weight_mean','weight_max','length_mean','height_mean','width_mean','purchase_month','purchase_weekday','promised_days']
cat=['customer_state','seller_state_first']; num=[x for x in features if x not in cat]
data=eligible.sort_values('order_purchase_timestamp').copy(); cut=int(len(data)*.8)
train,test=data.iloc[:cut],data.iloc[cut:]
pre=ColumnTransformer([('num',Pipeline([('impute',SimpleImputer(strategy='median'))]),num),('cat',Pipeline([('impute',SimpleImputer(strategy='most_frequent')),('onehot',OneHotEncoder(handle_unknown='ignore'))]),cat)])
tree=Pipeline([('pre',pre),('clf',DecisionTreeClassifier(max_depth=5,min_samples_leaf=200,class_weight='balanced',random_state=42))])
tree.fit(train[features],train.late)
prob=tree.predict_proba(test[features])[:,1]
pred=(prob>=.5).astype(int)
print('Entrenamiento:',len(train),'Prueba:',len(test),'tasa train/test:',train.late.mean(),test.late.mean())
print(classification_report(test.late,pred,zero_division=0))
print('PR-AUC:',average_precision_score(test.late,prob),'ROC-AUC:',roc_auc_score(test.late,prob))
print('Matriz de confusión:\n',confusion_matrix(test.late,pred))
print('Baseline siempre a tiempo:\n',classification_report(test.late,np.zeros(len(test),dtype=int),zero_division=0))
pd.DataFrame({'feature':tree.named_steps['pre'].get_feature_names_out(),'importance':tree.named_steps['clf'].feature_importances_}).sort_values('importance',ascending=False).head(15)

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4));sns.heatmap(confusion_matrix(test.late,pred),annot=True,fmt='d',ax=ax[0]);ax[0].set(xlabel='Predicción',ylabel='Real',title='Árbol, prueba temporal')
p,r,_=precision_recall_curve(test.late,prob);ax[1].plot(r,p);ax[1].axhline(test.late.mean(),ls='--',c='gray');ax[1].set(xlabel='Recall',ylabel='Precisión',title='Curva precisión sensibilidad');plt.tight_layout();plt.savefig('arbol_validacion.png',dpi=160);plt.show()

## K-Means
Segmentación descriptiva de órdenes con variables numéricas conocidas al comprar. No se usa la etiqueta al ajustar grupos. Se evalúan k=2..6 con silueta en una muestra reproducible. Los grupos son asociaciones, no causas ni perfiles estables fuera de la muestra.

In [ ]:
cluster_features=['n_items','price_sum','freight_sum','weight_mean','promised_days']
X=eligible[cluster_features]
scaler=Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler())])
Z=scaler.fit_transform(X)
rng=np.random.default_rng(42); ix=rng.choice(len(Z),min(2000,len(Z)),replace=False); Zs=Z[ix]
scores={k:silhouette_score(Zs,KMeans(n_clusters=k,n_init=10,random_state=42).fit_predict(Zs)) for k in range(2,7)}
best=max(scores,key=scores.get); km=KMeans(n_clusters=best,n_init=10,random_state=42).fit(Z)
eligible['cluster']=km.labels_
print('Siluetas:',scores,'k elegido:',best)
print(eligible.groupby('cluster')[cluster_features+['late']].agg(['count','median','mean']).round(3))

In [ ]:
summary=eligible.groupby('cluster').agg(orders=('order_id','size'),late_rate=('late','mean'),price_median=('price_sum','median'),freight_median=('freight_sum','median'),promised_median=('promised_days','median'))
summary.to_csv('resultados_segmentos.csv');summary[['late_rate']].plot.bar(title='Retraso observado por segmento');plt.tight_layout();plt.savefig('segmentos.png',dpi=160);plt.show()
pd.DataFrame([{'n_original':len(df),'n_elegible':len(eligible),'n_train':len(train),'n_test':len(test),'late_rate_test':test.late.mean(),'precision_late':classification_report(test.late,pred,output_dict=True,zero_division=0)['1']['precision'],'recall_late':classification_report(test.late,pred,output_dict=True,zero_division=0)['1']['recall'],'f1_late':classification_report(test.late,pred,output_dict=True,zero_division=0)['1']['f1-score'],'pr_auc':average_precision_score(test.late,prob),'roc_auc':roc_auc_score(test.late,prob),'k':best,'silhouette':scores[best]}]).to_csv('resultados_modelos.csv',index=False)

## Lectura responsable
Revisar métricas y gráficos producidos arriba antes de afirmar que el árbol tiene utilidad operativa. La precisión y la sensibilidad dependen del umbral y del costo de falsas alarmas; el estudio no observa costos reales. K-Means no predice retrasos. La evaluación excluye órdenes sin entrega observada y puede sesgarse por selección. No usar reseñas ni fechas posteriores a la compra como predictores.